# Notebook 07 — Phase 2: Classification & Ablation

## Objective
This notebook answers the Phase 2 subject questions:
1. Which features should be used for classification?
2. Which classifier to choose among NB, SVC, LogReg, MLP?
3. Does the classifier perform as well on docs as on queries?

## Structure
- **Section 0**: Setup & data loading
- **Section 1**: Feature extraction
- **Section 2**: Ablation — comparison of 12 combinations (3 features × 4 classifiers)
- **Section 3**: Analysis & choice of final model (justified by the ablation)
- **Section 4**: Final model — training, metrics, confusion matrix
- **Section 5**: Docs vs Queries

## Section 0 — Setup & Data Loading

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve project root
cwd = Path.cwd()
project_root = cwd.parent if (cwd.parent / 'src').exists() else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reproducibility — ALWAYS call in the first cell
from src.kaggle.submit_phase2 import set_global_seeds
from src.config import RANDOM_SEED
set_global_seeds(RANDOM_SEED)
print(f'Seeds fixed: {RANDOM_SEED}')

In [ ]:
from src.data.load import load_all
from src.data.preprocess import add_content_field

raw_dir = project_root / 'data' / 'raw'
docs, queries_train, queries_test, gts_raw = load_all(raw_dir)

docs, queries_train = add_content_field(docs, queries_train, clean=False)

print(f'Docs         : {len(docs)}')
print(f'Train queries: {len(queries_train)}')

## Section 1 — Feature Extraction

We extract texts and labels from docs + queries_train, then split into 3 stratified splits (train / val / test).

**Training data:** docs + queries_train
- Docs constitute the majority of the data (216k samples)
- queries_train provide short samples, close to the style of test queries → helps generalization

In [ ]:
from collections import Counter

from src.classification.features import (
    extract_texts_and_labels,
    stratified_split,
    build_tfidf_features,
    build_count_features,
    build_embedding_features,
)

texts, labels = extract_texts_and_labels(docs, queries_train)

X_train, y_train, X_val, y_val, X_test, y_test = stratified_split(
    texts, labels,
    val_size=0.15,
    test_size=0.15,
    random_state=RANDOM_SEED
)

print(f"Total examples: {len(texts)}")
print(f"  Train        : {len(X_train)} ({len(X_train)/len(texts)*100:.1f}%)")
print(f"  Val          : {len(X_val)}  ({len(X_val)/len(texts)*100:.1f}%)")
print(f"  Test         : {len(X_test)}  ({len(X_test)/len(texts)*100:.1f}%)")

print("\nLabel distribution (train):")
for cat, count in sorted(Counter(y_train).items()):
    print(f"  {cat:15s}: {count:>6}  ({count/len(y_train)*100:.1f}%)")

In [ ]:
# TF-IDF:
tfidf_vec, X_train_tfidf, X_val_tfidf, X_test_tfidf = build_tfidf_features(X_train, X_val, X_test)

# Count Vectorizer:
count_vec, X_train_count, X_val_count, X_test_count = build_count_features(X_train, X_val, X_test)

# Embeddings
_, X_train_emb, X_val_emb, X_test_emb = build_embedding_features(X_train, X_val, X_test)

print(f"TF-IDF  — train: {X_train_tfidf.shape}, val: {X_val_tfidf.shape}, test: {X_test_tfidf.shape}")
print(f"Count   — train: {X_train_count.shape}, val: {X_val_count.shape}, test: {X_test_count.shape}")
print(f"Embeddings — train: {X_train_emb.shape}, val: {X_val_emb.shape}, test: {X_test_emb.shape}")

## Section 2 — Ablation

We test all combinations: **3 features × 4 classifiers = 12 combinations**

Evaluation on the **val set** for each combination.

Note: MultinomialNB is incompatible with embeddings (negative values).

In [ ]:
from src.classification.model import Classifier
from src.evaluation.metrics import accuracy, macro_f1, balanced_accuracy

features_map = {
    "tfidf":     (X_train_tfidf,  X_val_tfidf),
    "count":     (X_train_count,  X_val_count),
    "embedding": (X_train_emb,    X_val_emb)
}
classifiers = ["nb", "svc", "logreg", "mlp"]
clf_kwargs = {
    "nb":     {},
    "svc":    {},
    "logreg": {"max_iter": 1000},
    "mlp":    {}
}

results = []

for feat_name, (X_train_feat, X_val_feat) in features_map.items():
    for clf_name in classifiers:
        if clf_name == "nb" and feat_name == "embedding":
            continue
        
        # Initialize the classifier:
        clf = Classifier(method=clf_name, **clf_kwargs[clf_name])

        # Train the classifier on the training set:
        clf.fit(X_train_feat, y_train)

        # Predict on the validation set:
        y_pred = clf.predict(X_val_feat)

        acc = accuracy(y_true= y_val, y_pred= y_pred)
        mac = macro_f1(y_true= y_val, y_pred= y_pred)
        bal = balanced_accuracy(y_true= y_val, y_pred= y_pred)

        results.append({
            "features":         feat_name,
            "classifier":       clf_name,
            "accuracy":         acc,
            "macro_f1":         mac,
            "balanced_accuracy": bal
        })

        print(f"[{feat_name} × {clf_name}] macro_f1={mac:.4f}, accuracy={acc:.4f}, balanced_accuracy={bal:.4f}.")

In [ ]:
pd.DataFrame(results).sort_values("macro_f1", ascending=False)

## Section 3 — Analysis & Choice of Final Model

### Observations

**Which feature method gives the best results?**
TF-IDF clearly dominates: the top 3 spots in the table are all TF-IDF.
Count Vectorizer follows closely (~0.002 macro_f1 gap), which is expected — TF-IDF
simply adds log(1+tf) weighting that attenuates very frequent terms.
Embeddings alone are the least performant on this dataset (max macro_f1 = 0.9695),
probably because 384 dimensions capture less discriminative signal than
50,000 n-grams on such specialized texts (code, commands, LaTeX...).

**Which classifier is the most performant?**
LinearSVC is the best on TF-IDF and Count (macro_f1=0.9852 and 0.9793).
For embeddings, MLP takes the lead (macro_f1=0.9695) — consistent, dense embeddings
are better exploited by a non-linear network.
NaiveBayes is systematically last on bag-of-words features.

**Is there a good performance/speed trade-off?**
Yes: TF-IDF × SVC. LinearSVC is near-instantaneous to train on sparse matrices,
and it is also the best model. MLP is significantly slower for no gain on TF-IDF.
LogReg is a good second (−0.006 macro_f1) if calibrated probabilities are needed.

### Final Model Choice

**→ TF-IDF × LinearSVC** (`macro_f1=0.9852`, `accuracy=0.9870`, `balanced_accuracy=0.9848`)

Justification:
- Best macro_f1 on the val set, by a clear margin (+0.006 over second place)
- Very fast to train (sparse matrix + linear SVM)
- Only drawback: no `predict_proba` — but for reranking (Person B),
  LogReg can be used if probabilities are needed

## Section 4 — Final Model

We train the model chosen from the ablation and evaluate it on the **test set**.

In [ ]:
from scipy.sparse import vstack
from src.evaluation.metrics import classification_report_phase2

BEST_FEAT = "tfidf"
BEST_CLF = "svc"

# Concatenate train and val sets:
y_trainval = y_train + y_val
X_trainval_tfidf = vstack([X_train_tfidf, X_val_tfidf])

# Initialize the best classifier:
clf = Classifier(method=BEST_CLF, **clf_kwargs[BEST_CLF])

# Train the classifier on the combined train + val set:
clf.fit(X= X_trainval_tfidf, y= y_trainval)

y_pred = clf.predict(X_test_tfidf)

acc = accuracy(y_true= y_test, y_pred= y_pred)
mac = macro_f1(y_true= y_test, y_pred= y_pred)
bal = balanced_accuracy(y_true= y_test, y_pred= y_pred)
cla = classification_report_phase2(y_true= y_test, y_pred= y_pred)

print(f"[{BEST_FEAT} × {BEST_CLF}] macro_f1={mac:.4f}, accuracy={acc:.4f}, balanced_accuracy={bal:.4f}. classification_report_phase2:\n{cla}")

In [ ]:
from src.evaluation.metrics import confusion_matrix_phase2
from src.classification.interfaces import CATEGORIES

cm = confusion_matrix_phase2(y_pred= y_pred, y_true= y_test, labels= CATEGORIES)

plt.imshow(cm, cmap= "Blues")
plt.colorbar()
plt.title("Confusion Matrix — TF-IDF × LinearSVC (test set)")

plt.xticks(range(5), CATEGORIES, rotation=45, ha="right")
plt.yticks(range(5), CATEGORIES)

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(len(CATEGORIES)):
    for j in range(len(CATEGORIES)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

**Confusion Matrix Analysis — TF-IDF × LinearSVC (test set)**

**Well-classified categories:**
- `tex` is near-perfect (precision=1.00, recall=1.00) — the LaTeX vocabulary is
  extremely distinctive (`\begin`, `\frac`, `\documentclass`... commands) and
  does not appear in any other category.
- `gaming` is very well classified (f1=0.99) — the video game lexical field
  (game names, "fps", "console", "achievements"...) is unambiguous.

**Recurring confusions:**
- The few errors concentrate on `android`, `programmers` and `unix`,
  which share common technical vocabulary: shell commands, Python/Java code,
  package references. A post about Android programming may look like
  a general programming post.
- `unix` and `programmers` are most likely to be confused: bash scripting questions,
  file or process management questions can belong to either category.

**Conclusion:** The model is robust — overall error rate is below 1.5%.
The rare confusions are semantically justified and difficult to avoid even for
a human.

## Section 5 — Docs vs Queries

The classifier is trained on docs (long texts) but must predict the category of queries (short texts). Does it generalize well?

In [ ]:
docs_texts, docs_labels = extract_texts_and_labels(docs= docs)
queries_texts, queries_labels = extract_texts_and_labels(docs= [], queries= queries_train)

X_docs = tfidf_vec.transform(docs_texts)
X_queries = tfidf_vec.transform(queries_texts)

print(f"Docs    : {X_docs.shape}, labels: {len(docs_labels)}")
print(f"Queries : {X_queries.shape}, labels: {len(queries_labels)}")

In [ ]:
from src.evaluation.evaluate import evaluate_docs_vs_queries

results_dvq = evaluate_docs_vs_queries(clf= clf, X_docs= X_docs, y_docs= docs_labels, X_queries= X_queries, y_queries= queries_labels)

print("=== DOCS ===")
print(results_dvq["docs"]["classification_report"])
print("=== QUERIES ===")
print(results_dvq["queries"]["classification_report"])

## Analysis — Docs vs Queries

**Does the classifier perform as well on queries as on docs?**

Yes, and remarkably so:
- **Docs**: accuracy=1.00, macro_f1=1.00 — near-perfect (expected, the model was trained on docs)
- **Queries**: accuracy=0.99, macro_f1=0.99 — near-identical despite the difference in length

The gap is below 0.01 on all metrics, which is excellent.

**Why is the gap so small?**

Two main reasons:
1. **The categories are semantically very distinct** — even a short query like
   *"how to install a package on ubuntu"* contains enough signal to be
   classified as `unix` without ambiguity.
2. **queries_train were included in the training data** — the model has seen
   short samples during fitting, which helps it generalize to test queries.

**Which categories are most problematic on queries?**

- `android`: recall=0.96 — a few android queries are misclassified (probably
  confused with `programmers`, as Android dev questions look like general
  programming questions).
- `unix`: precision=0.96 — a few queries from other categories are wrongly predicted
  as `unix` (likely `programmers` queries containing shell commands).

**Conclusion:** The model generalizes very well from documents to queries.
Including queries_train in the training data is a justified decision
that helps generalization without visible overfitting.